# 02 — Modeling & Hyperparameter Tuning

**Owner**: Isaac  
**Goal**: Train and tune the four core anomaly detection models (Isolation Forest, One-Class SVM, LOF, Elliptic Envelope).

See `TASKS_ISAAC.md` for the detailed checklist.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("../").resolve()))

In [ ]:
import pandas as pd
import numpy as np

from src.models import (
    train_isolation_forest,
    train_one_class_svm,
    train_lof,
    train_elliptic_envelope,
)

print("Imports successful")

In [ ]:
df = pd.read_csv("../ai4i2020.csv")

print(df.shape)

df.head()

In [ ]:
df.columns

In [ ]:
from src.preprocessing import (
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    split_features_and_labels,
    build_preprocessor,
)

# Use Noah's shared preprocessor for consistency with the EDA rationale:
# StandardScaler on the 5 numeric sensors + OneHotEncoder on Type
# (drop="first" to avoid perfect collinearity).
# Avoiding LabelEncoder, which would impose an arbitrary order on L/M/H
# and bias the Euclidean distances used by LOF / OC-SVM / Elliptic Envelope.
X_raw, _y_held_out = split_features_and_labels(df)
preprocessor = build_preprocessor()
X_scaled = preprocessor.fit_transform(X_raw)

print(f"X_scaled shape: {X_scaled.shape}")
print(f"Features: {list(preprocessor.get_feature_names_out())}")

In [ ]:
iso_model = train_isolation_forest(
    X_scaled,
    contamination=0.034,
    n_estimators=200,
    max_samples="auto",
    random_state=42,
)

iso_preds = iso_model.predict(X_scaled)

unique, counts = np.unique(iso_preds, return_counts=True)
dict(zip(unique, counts))

In [ ]:
from pathlib import Path

results_dir = Path("../outputs/results")
results_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame({"prediction": iso_preds}).to_csv(
    results_dir / "preds_isolation_forest.csv",
    index=True,
)

print("Export OK")

In [ ]:
ocsvm_model = train_one_class_svm(
    X_scaled,
    nu=0.034,
    kernel="rbf",
    gamma="scale",
)

ocsvm_preds = ocsvm_model.predict(X_scaled)

unique, counts = np.unique(ocsvm_preds, return_counts=True)
dict(zip(unique, counts))

In [ ]:
pd.DataFrame({"prediction": ocsvm_preds}).to_csv(
    results_dir / "preds_ocsvm.csv",
    index=True,
)

print("OCSVM export OK")

In [ ]:
lof_model = train_lof(
    X_scaled,
    n_neighbors=35,
    contamination=0.034,
)

lof_preds = lof_model.fit_predict(X_scaled)

unique, counts = np.unique(lof_preds, return_counts=True)
dict(zip(unique, counts))

In [ ]:
pd.DataFrame({"prediction": lof_preds}).to_csv(
    results_dir / "preds_lof.csv",
    index=True,
)

print("LOF export OK")

In [ ]:
elliptic_model = train_elliptic_envelope(
    X_scaled,
    contamination=0.034,
    support_fraction=None,
    random_state=42,
)

elliptic_preds = elliptic_model.predict(X_scaled)

unique, counts = np.unique(elliptic_preds, return_counts=True)
dict(zip(unique, counts))

In [ ]:
pd.DataFrame({"prediction": elliptic_preds}).to_csv(
    results_dir / "preds_elliptic.csv",
    index=True,
)

print("Elliptic export OK")

In [ ]:
models_dict = {
    "isolation_forest": iso_model,
    "ocsvm": ocsvm_model,
    "lof": lof_model,
    "elliptic": elliptic_model,
}

print(models_dict.keys())

## Hyperparameter tuning strategy

We test several hyperparameter configurations for each anomaly detection model.  
Since the dataset contains approximately 3.4% anomalies, contamination-like parameters are tested around this value: 0.02, 0.034, 0.05, and 0.10.

The objective is not only to maximize accuracy, but also to check whether the models produce stable and coherent anomaly rates.

In [ ]:
summary = pd.DataFrame({
    "model": [
        "Isolation Forest",
        "One-Class SVM",
        "LOF",
        "Elliptic Envelope",
    ],
    "anomalies_detected": [
        (iso_preds == -1).sum(),
        (ocsvm_preds == -1).sum(),
        (lof_preds == -1).sum(),
        (elliptic_preds == -1).sum(),
    ],
    "normal_points": [
        (iso_preds == 1).sum(),
        (ocsvm_preds == 1).sum(),
        (lof_preds == 1).sum(),
        (elliptic_preds == 1).sum(),
    ],
})

summary["anomaly_rate"] = summary["anomalies_detected"] / len(X_scaled)

summary

## First modeling results

All four models were calibrated around the observed anomaly rate of approximately 3.4%.  
The predictions follow the sklearn convention: `-1` for anomalies and `1` for normal observations.

Isolation Forest, One-Class SVM, LOF, and Elliptic Envelope all detect around 340 anomalies, which is coherent with the expected anomaly proportion.

However, these models rely on different assumptions:
- Isolation Forest isolates anomalies through random partitions.
- One-Class SVM learns a non-linear boundary around normal observations.
- LOF detects local density deviations.
- Elliptic Envelope assumes a multivariate Gaussian distribution, which may be restrictive for this dataset.

In [ ]:
iso_results = []

for contamination in [0.02, 0.034, 0.05, 0.10]:
    for n_estimators in [100, 200, 300]:
        for max_samples in ["auto", 0.5]:
            
            model = train_isolation_forest(
                X_scaled,
                contamination=contamination,
                n_estimators=n_estimators,
                max_samples=max_samples,
                random_state=42,
            )
            
            preds = model.predict(X_scaled)
            
            iso_results.append({
                "model": "Isolation Forest",
                "contamination": contamination,
                "n_estimators": n_estimators,
                "max_samples": max_samples,
                "anomalies_detected": (preds == -1).sum(),
                "anomaly_rate": (preds == -1).mean(),
            })

iso_results_df = pd.DataFrame(iso_results)
iso_results_df

In [ ]:
ocsvm_results = []

for nu in [0.02, 0.034, 0.05, 0.10]:
    for gamma in ["scale", "auto", 0.01, 0.1, 1.0]:
        
        model = train_one_class_svm(
            X_scaled,
            nu=nu,
            kernel="rbf",
            gamma=gamma,
        )
        
        preds = model.predict(X_scaled)
        
        ocsvm_results.append({
            "model": "One-Class SVM",
            "nu": nu,
            "gamma": gamma,
            "anomalies_detected": (preds == -1).sum(),
            "anomaly_rate": (preds == -1).mean(),
        })

ocsvm_results_df = pd.DataFrame(ocsvm_results)
ocsvm_results_df

In [ ]:
lof_results = []

for n_neighbors in [10, 20, 35, 50]:
    for contamination in [0.02, 0.034, 0.05, 0.10]:
        
        model = train_lof(
            X_scaled,
            n_neighbors=n_neighbors,
            contamination=contamination,
        )
        
        preds = model.fit_predict(X_scaled)
        
        lof_results.append({
            "model": "LOF",
            "n_neighbors": n_neighbors,
            "contamination": contamination,
            "anomalies_detected": (preds == -1).sum(),
            "anomaly_rate": (preds == -1).mean(),
        })

lof_results_df = pd.DataFrame(lof_results)
lof_results_df

In [ ]:
elliptic_results = []

for contamination in [0.02, 0.034, 0.05, 0.10]:
    for support_fraction in [None, 0.75, 0.9]:
        
        model = train_elliptic_envelope(
            X_scaled,
            contamination=contamination,
            support_fraction=support_fraction,
            random_state=42,
        )
        
        preds = model.predict(X_scaled)
        
        elliptic_results.append({
            "model": "Elliptic Envelope",
            "contamination": contamination,
            "support_fraction": support_fraction,
            "anomalies_detected": (preds == -1).sum(),
            "anomaly_rate": (preds == -1).mean(),
        })

elliptic_results_df = pd.DataFrame(elliptic_results)
elliptic_results_df

## Modeling interpretation

The selected contamination level is `0.034`, because it is consistent with the anomaly rate observed in the dataset.

For Isolation Forest, `n_estimators=200` is retained as a compromise between computation time and model stability.

For One-Class SVM, the RBF kernel is used because the anomaly boundary is unlikely to be purely linear. `gamma="scale"` is retained because it adapts to the variance of the scaled features.

For LOF, `n_neighbors=35` is used as a compromise between very local neighborhoods, which can be noisy, and larger neighborhoods, which can smooth subtle anomalies.

For Elliptic Envelope, the model is kept as a useful baseline, but its main limitation is that it assumes a multivariate Gaussian distribution. This assumption may be partially violated in the AI4I dataset, so its results should be interpreted carefully.

In [ ]:
results_dir = Path("../outputs/results")
results_dir.mkdir(parents=True, exist_ok=True)

predictions = {
    "isolation_forest": iso_preds,
    "ocsvm": ocsvm_preds,
    "lof": lof_preds,
    "elliptic": elliptic_preds,
}

for name, preds in predictions.items():
    pd.DataFrame({"prediction": preds}).to_csv(
        results_dir / f"preds_{name}.csv",
        index=True,
    )

print("All prediction files exported.")

In [ ]:
models_dict = {
    "isolation_forest": iso_model,
    "ocsvm": ocsvm_model,
    "lof": lof_model,
    "elliptic": elliptic_model,
}

models_dict.keys()